In [1]:
import sys
sys.path.append("..")

from pathlib import Path
from sentence_transformers import SentenceTransformer

from src.data import load_dataset
from src.embedder import record_to_text, get_embeddings, save_embeddings, load_embeddings, get_cache_path
from src.search import search_all_questions
from src.metrics import evaluate
from src.config import MODELS, TOP_K

corpus, questions, categories = load_dataset()
corpus_texts = [record_to_text(item) for item in corpus]

print(f"Корпус: {len(corpus)} фрагментов")
print(f"Вопросы: {len(questions)} вопросов")

Корпус: 200 фрагментов
Вопросы: 25 вопросов


In [2]:
model_name = MODELS[2]
cache_path = get_cache_path(model_name)

if Path(cache_path).exists():
    corpus_embedding = load_embeddings(cache_path)
    print("Загруженна из кэша")
else:
    corpus_embedding = get_embeddings(corpus_texts, model_name)
    save_embeddings(cache_path, corpus_embedding)
    print("Вычислено и добавлено в кэш")

Загруженна из кэша


In [3]:
model = SentenceTransformer(model_name)
search_outputs = search_all_questions(questions, model, corpus, corpus_embedding)

metrics = evaluate(search_outputs, k=TOP_K)
print(metrics)
print("Для модели:", model_name)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

{'precision_at_3': 0.96, 'mrr': 0.7733333333333333, 'num_questions': 25}
Для модели: intfloat/multilingual-e5-small


In [4]:
retrieved_ids_list = []
for search_result in search_outputs:
    top_ids = [result["id"] for result in search_result["results"][:TOP_K]]
    retrieved_ids_list.append(top_ids)

In [5]:
errors = []
for i, (q, retrieved_ids) in enumerate(zip(questions, retrieved_ids_list)):
    correct_id = q["correct_chunk_id"]
    if correct_id not in retrieved_ids:
        correct_func = next((item for item in corpus if item["id"] == correct_id), {})
        wrong_funcs = [next((item for item in corpus if item["id"] == rid), {}) for rid in retrieved_ids]
        
        errors.append({
            "question_id": q["question_id"],
            "query": q["query"],
            "query_lang": q.get("language", "unknown"),
            "correct_func_name": correct_func.get("function_name", "unknown"),
            "correct_category": correct_func.get("category", "unknown"),
            "retrieved_names": [f.get("function_name", "unknown") for f in wrong_funcs],
            "retrieved_categories": [f.get("category", "unknown") for f in wrong_funcs],
        })

print(f"Всего вопросов: {len(questions)}")
print(f"Ошибок: {len(errors)}")

if errors:
    print("\n" + "="*80)
    print("ТАБЛИЦА ОШИБОК")
    print("="*80)
    
    for err in errors:
        print(f"\n Вопрос {err['question_id']}: {err['query']}")
        print(f"   Должно быть: {err['correct_func_name']} ({err['correct_category']})")
        print(f"   Нашлось: {err['retrieved_names']} ({err['retrieved_categories']})")

Всего вопросов: 25
Ошибок: 1

ТАБЛИЦА ОШИБОК

 Вопрос q_08: массовая вставка большого количества записей
   Должно быть: bulkInsertRecords (database)
   Нашлось: ['chunkList', 'chunk_list', 'find_orders_by_user'] (['utils', 'utils', 'database'])


In [6]:
import json
from pathlib import Path

outputs_dir = Path.cwd().parent / "outputs"
outputs_dir.mkdir(exist_ok=True) 

output_path_e5 = outputs_dir / "errors_e5.json"
with open(output_path_e5, "w", encoding="utf-8") as f:
    json.dump(errors, f, ensure_ascii=False, indent=2)
print(f"Ошибки сохранены в {output_path_e5}")

Ошибки сохранены в C:\CPP\semantic-search-case\outputs\errors_e5.json
